# **Rate My Recipe**

## 1. Recipe Characteristics and Ratings

This notebook explores whether recipe characteristics, such as preparation time and number of ingredients, are associated with user ratings.

The analysis focuses on:
- Preparation time.
- Number of ingredients.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from recipe_project.data_loader import load_recipes, load_interactions
from recipe_project.cleaning import filter_valid_interactions
from recipe_project.analysis import calculate_recipe_stats

In [3]:
recipes = load_recipes()
interactions = load_interactions()

In [4]:
valid_interactions = filter_valid_interactions(interactions)
recipe_stats = calculate_recipe_stats(valid_interactions)

In [5]:
print("Valid interactions:", valid_interactions.shape)
print("Recipe statistics:", recipe_stats.shape)

display(recipe_stats.head())

Valid interactions: (1071520, 5)
Recipe statistics: (226590, 3)


,recipe_id,rating_count,average_rating
0,38,4,4.250000
1,39,1,3.000000
2,40,9,4.333333
3,41,2,4.500000
4,43,1,1.000000


## 2. Preparation Time

This section explores the distribution of recipe preparation times and their relationship with average user ratings.

### 2.1 Preparation Time Distribution

This section examines the distribution of recipe preparation times.

In [6]:
pd.set_option("display.float_format", "{:,.2f}".format)
recipes["minutes"].describe()

count         231,637.00
mean            9,398.55
std         4,461,963.04
min                 0.00
25%                20.00
50%                40.00
75%                65.00
max     2,147,483,647.00
Name: minutes, dtype: float64

In [7]:
recipes["minutes"].value_counts().head(15)

minutes
30    18391
20    14822
40    13772
25    13204
35    12385
15    12213
45    11517
10    11291
60    10302
50     9929
5      9449
55     7173
75     6267
70     6154
65     5495
Name: count, dtype: int64

Recipes are grouped into four categories:

- **Quick:** up to 30 minutes.
- **Medium:** 31–60 minutes.
- **Long:** 61–90 minutes.
- **Very Long:** over 90 minutes.

This classification allows us to compare average ratings across different preparation times.

In [8]:
preparation_categories = []

for minutes in recipes["minutes"]:
    if pd.isna(minutes):
        preparation_categories.append("Unknown")
    elif minutes <= 30 and minutes >= 0:
        preparation_categories.append("Quick")
    elif minutes <= 60:
        preparation_categories.append("Medium")
    elif minutes <= 90:
        preparation_categories.append("Long")
    else:
        preparation_categories.append("Very Long")

recipes["preparation_category"] = preparation_categories

recipes["preparation_category"].value_counts()

preparation_category
Quick        99053
Medium       70077
Very Long    33436
Long         29071
Name: count, dtype: int64

In [9]:
recipe_analysis = recipes.merge(
    recipe_stats,
    left_on="id",
    right_on="recipe_id",
    how="inner"
)

In [10]:
category_stats = recipe_analysis.groupby("preparation_category").agg(
    recipe_count=("id", "count"),
    average_rating=("average_rating", "mean"),
    average_rating_count=("rating_count", "mean")
)

display(category_stats)

,recipe_count,average_rating,average_rating_count
preparation_category,,,
Long,28352,4.58,4.86
Medium,68495,4.56,4.75
Quick,97250,4.60,4.57
Very Long,32493,4.57,5.05


### 2.2 Preparation Time and Bayesian Ranking

This section compares recipe rankings across preparation time categories using the Bayesian Weighted Ranking.

In [11]:
from recipe_project.ranking import bayesian_weighted_ranking

weighted_ranking = bayesian_weighted_ranking(recipe_stats)

In [12]:
ranking_by_category = weighted_ranking.merge(
    recipes[["id", "preparation_category"]],
    left_on="recipe_id",
    right_on="id",
    how="inner"
)

In [13]:
top_10 = ranking_by_category.nlargest(10, "bayesian_score")

category_distribution = (
    top_10["preparation_category"].value_counts()
)

display(category_distribution)

preparation_category
Quick        5
Medium       2
Very Long    2
Long         1
Name: count, dtype: int64

In [14]:
top_100 = ranking_by_category.nlargest(100, "bayesian_score")

category_distribution_100 = (
    top_100["preparation_category"].value_counts()
)

display(category_distribution_100)

preparation_category
Quick        43
Medium       25
Long         16
Very Long    16
Name: count, dtype: int64

**Observations**

Quick recipes are the most represented both in the top 10 and in the 100 Bayesian ranking. However, this result should be interpreted considering the different sizes of the preparation time categories.

In [16]:
overall_distribution = (
    ranking_by_category["preparation_category"]
    .value_counts(normalize=True) * 100
)

display(overall_distribution.round(2))

preparation_category
Quick       42.92
Medium      30.23
Very Long   14.34
Long        12.51
Name: proportion, dtype: float64

**Observations**

- Quick recipes represent 43% of the top 100, almost identical to their 42.92% share in the overall ranking.
- Medium recipes are less represented in the top 100 (25% vs. 30.23%).
- Long and Very Long recipes are slightly more represented in the top 100.

**Conclusion**

Preparation time does not appear to strongly influence the Bayesian ranking. Quick recipes are the most numerous in the dataset, but they are not particularly overrepresented among the top 100 recipes.